# 骨龄评估系统 — 开发总结（截至 2026-08-06）

> 目标：读取左手腕 X 光片 → 输出骨龄（岁/月）。采用「检测 → 骨过滤 → 分类 → RUS 计分 → 骨龄」两阶段方案。

## 1. 数据准备
- `voc_to_yolo.py`：把 VOC 格式的 881 张手掌标注（7 类骨骼）转成 YOLO 格式，训练/验证 705/176（无类别泄漏）
- `prepare_classification.py`：构建 ImageFolder 分类结构，每个等级至少 4 张验证样本
- `data_audit.py`：数据质检（亮度/重复图/等级可分性），发现并删除 2 张标签冲突的桡骨图

## 2. 预处理（`preprocess.py`）
- 灰度 + 中值滤波(3) + **CLAHE 自适应直方图均衡化**（clip=2.0, grid=8×8）

## 3. 检测模型（YOLOv8n）
- `train_detection.py` 两阶段迁移学习（先冻结 10 层训 60 epoch，再全量微调 40 epoch）
- **mAP50 = 0.991**，P=0.996 R=0.994；权重 `runs/bone7_ft/weights/best.pt`

## 4. 分类模型（9 个 ResNet18）
- 每个关节一个分类器（DIP/DIPFirst/MCP/MCPFirst/MIP/PIP/PIPFirst/Radius/Ulna）
- 类权重 CE + 余弦退火 + 早停；**关键修复**：ImageFolder 按字符串排序导致等级错位，改用 `grade_list` 数值映射
- **Ulna 改用 CORN 序数回归**：MAE 2.48 → 1.67（-33%）；其余 7 关节 ordinal 训练进行中
- **类别不平衡**（各关节 4x~20x，见下方统计 cell）：用类权重 CE（`w_i = total/(n_class·count_i)`）+ 按等级分层划分（每等级 ≥4 张进 val、≥1 张进 train）处理；无 <2 张的极端等级

## 5. 骨过滤（`filter_bones.py`）
- 从 7 类检测结果选出 **RUS 13 块骨**（Radius/Ulna/MCP-1,3,5/PIP-1,3,5/MIP-3,5/DIP-1,3,5）
- 拇指侧判定 + 顺序保持匹配，验证 30/30 完整

## 6. 计分模块（`scoring.py`）
- 解析用户提供的官方表格：`骨发育等级对照表.csv`（RUS-CHN 计分）+ `骨龄评分参考表_TW3_RUS系列.csv`（骨龄换算）
- `bone_age_from_rus()` 返回（中值/下限/上限），越界截断

## 7. 端到端流水线（`pipeline.py`）
- 检测 → 13 骨过滤 → 分类 → RUS 计分 → 骨龄可视化

## 8. RSNA 验证 → 发现问题 → 数据驱动校准
- 用 RSNA 儿科骨龄挑战赛真实标签验证：881 张训练图全部来自该数据集
- **RUS 表硬查失败**：MAE = 53.5 月、相关 -0.32
  - 根因：arthrosis 部分骨等级标注与真实成熟度脱节（桡骨相关仅 +0.15，却是最大权重 210/1000，成为噪声）
- **数据驱动校准修复**（`calibrate.py`）：13 骨 RUS 得分特征 → GradientBoosting 回归骨龄
  - 2306 张标注图按年龄分层 85/15
  - **测试 MAE = 13.22 月（1.10 岁），相关 0.902**（Ridge 22.2 月）
  - 模型 `models/bone_age_regressor.pkl`；pipeline 新增 `--calibrated` 生产模式
  - 实测：14732 真实 5.8 岁 → 校准预测 6.11 岁（误差 3.7 月）；RUS 表预测 10.5 岁（误差 56 月）

## 9. Git 进度
- `02cdac4` 全链路代码、`54502c9` TW3 表接入、`7aba69a` 数据驱动校准 — 均已推送 GitHub

## 下一步
- 等 8 关节 ordinal 训练完成（PIPFirst/Radius 待出）→ 评估是否全量切换 ordinal
- 等 RSNA 训练集图片下载完 → 扩充校准数据进一步降 MAE
- 全量 1425 验证集正式评估报告


In [1]:
# 类别分布统计：检查分类数据是否存在类别不平衡
# 处理措施：类权重 CE (w_i = total/(n_class*count_i)) + 按等级分层划分
from pathlib import Path

base = Path("Bone Age Assessment/datasets/classification_pre")
rows = []
for joint in sorted(p.name for p in base.iterdir() if p.is_dir()):
    c = {}
    for split in ["train", "val"]:
        d = base / joint / split
        for g in sorted(int(x.name) for x in d.iterdir() if x.is_dir()):
            n = len(list((d / str(g)).glob("*.png")))
            c[g] = c.get(g, 0) + n
    grades = sorted(c)
    cnts = [c[g] for g in grades]
    mx, mn, total = max(cnts), min(cnts), sum(cnts)
    low = [g for g in grades if c[g] < 2]
    rows.append((joint, total, len(grades), mn, mx, mx / mn, low))

print(f"{'关节':10s} {'总数':>5s} {'等级数':>4s} {'最少':>4s} {'最多':>4s} {'倍率':>7s}  <2张等级")
for joint, total, ng, mn, mx, ratio, low in rows:
    print(f"{joint:10s} {total:5d} {ng:4d} {mn:4d} {mx:4d} {ratio:6.1f}x  {low}")

关节            总数  等级数   最少   最多      倍率  <2张等级
DIP         1262   11   24  273   11.4x  []
DIPFirst     635   11    9  154   17.1x  []
MCP         1262   10   28  211    7.5x  []
MCPFirst     633   11   29  115    4.0x  []
MIP         1262   12   20  353   17.6x  []
PIP         1264   12   12  243   20.2x  []
PIPFirst     635   12   14  117    8.4x  []
Radius       646   14   15  154   10.3x  []
Ulna         632   12   20  189    9.4x  []
